# 11 HCA*：层次协作 A*（优先规划）


本课按百科条目写。HCA\* 是仓储/游戏里最常见的多车实用算法：固定全序，后车把前车当移动障碍。调度台在最后。

---

## 1. 定义

**HCA\***（Hierarchical Cooperative A\*，Silver, AIIDE 2005）= 固定全序上的优先规划：

1. 按 `order` 处理每台车。
2. 对该车做 **时间 A\***（08 课），占用表里已有更高优先级的路径。
3. 成功则把路径写入占用表（顶点+边）。

WHCA\* 是其窗口版（下一课）。FIFO 调度 ≈ 用「谁先请求」当 order 的 HCA\*。

- 输入：图、起终点、全序。
- 输出：每车一条时空路径，或某台失败。

---

## 2. 保证（务必读）

- **不必最优**：顺序差，SOC 会差。先走堵路口的车，后面全绕。
- **不完全**：有向窄廊对头时，两种全序都可能失败（双方都要对方先让，但先走的车占死通道）。这不是实现 bug，是算法能力边界。
- 无向网格、空位足够、允许等待时，许多实例能行。

对比：联合 A\* 最优但爆；CBS 最优且常更快；HCA 最快但不保证。

---

## 3. 不变量

- 处理完前 i 台后，这 i 条路径彼此无顶点/边冲突（后车看见了前车的表）。
- 预约必须锁边，否则对向擦肩漏检。
- 终点是否 **无限 stay**：库默认只锁路径上的有限拍。交换任务时若把终点锁到 horizon 末尾，会把别人的起点占死。

---

## 4. 伪代码

```
table ← 空
for agent in order:
    π ← TimeA*(start[a], goal[a], table)
    if 失败: return FAILURE
    Reserve(table, π)
return {π}

Reserve:
    对路径每个 (v,t): occupy 顶点
    对每步 (v, v', t): occupy_edge
```

---

## 5. 手算

开网格两车各走一行，互不相交：任意序都成功，SOC = 两段曼哈顿之和。

有向 `L→M→R`，a 在 L 要去 R，b 在 R 要去 L：b 根本没有出边离开 R，HCA 无论谁先都失败。联合 A\* / CBS 也失败——图本身无解。HCA 的不完全性出现在 **无向可解、但错误优先级把通道占死** 的图上；教学负例用有向死锁即可。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：外层循环（底层复用 08 课时间 A*）


In [ ]:
from evals.scenarios import two_agent_tunnel
from lib.reservation import ReservationTable
from lib.algos.temporal import time_astar
from lib.algos.prioritized import _reserve_path
from lib.agents import is_conflict_free, soc
from lib.maps import directed_deadlock_map
from lib.pretty import draw_grid


def hca_simple(graph, starts, goals, order):
    table = ReservationTable()
    paths = {}
    for agent in order:
        res = time_astar(graph, starts[agent], goals[agent], table=table, agent=agent)
        print(agent, "found", res.found, "cost", res.cost, "path", res.path)
        if not res.found:
            return {"found": False, "paths": paths, "order": order}
        paths[agent] = res.path
        _reserve_path(table, res.path, agent)
    return {"found": is_conflict_free(paths), "paths": paths, "soc": soc(paths), "order": order}


g, starts, goals = two_agent_tunnel()
print(draw_grid(g.width, g.height, start=starts["a"], goal=goals["a"]))
print("a", starts["a"], "->", goals["a"], "b", starts["b"], "->", goals["b"])
print("序 a,b", hca_simple(g, starts, goals, ["a", "b"])["found"])
print("序 b,a", hca_simple(g, starts, goals, ["b", "a"])["found"])


## 7. 逐行

| 行 | 作用 |
|----|------|
| 同一 `table` 贯穿 | 后车看见前车预约 |
| `time_astar(..., agent=agent)` | 本人的锁对自己 free |
| `_reserve_path` | 顶点+边整条锁上 |
| `is_conflict_free` | 用 stay-at-target pad 再验一次 |

错误写法：每台车 `table = ReservationTable()` ——等于没有协作。


In [ ]:
from lib.algos.prioritized import hca

r = hca(g, starts, goals, order=["a", "b"])
print("库 HCA found", r["found"], "SOC", r["soc"], "order", r["order"])

dead = directed_deadlock_map()
print(
    "有向死锁",
    hca(dead, {"a": "L", "b": "R"}, {"a": "R", "b": "L"})["found"],
)


## 常见 bug

1. 预约时不锁边。
2. 终点无限 stay 占死别人起点（交换任务经典坑）。
3. 每台车一张新表。
4. 底层不用时间维、不能等待。
5. 把失败当成「地图无解」——先换个 order 再下结论。

## 条目小结

| 项目 | 内容 |
|------|------|
| 思想 | 全序 + 时间 A\* + 占用表 |
| 最优 | 否 |
| 完备 | 否（有向/窄廊） |
| 下一课 | 只保证未来 w 拍 |

## 练习

1. 把 `_reserve_path` 改成只 occupy 顶点，造一个对向擦肩的反例。
2. 为何 FIFO（谁先叫车谁优先）在仓储里够用、在对换任务里不够？
3. 三车全序有 6 种。HCA 会试几种？PBS 下一课怎么处理。


In [ ]:
from lib.studio.widget import PlannerStudio
studio = PlannerStudio()
studio.show_multi(g.to_studio(), r, "HCA*")
studio
